In [1]:
%pip -q install ultralytics==8.4.1 pycocotools==2.0.11 tabulate
!git clone https://github.com/mdmohsin212/WasteScope.git

Cloning into 'WasteScope'...
remote: Enumerating objects: 400, done.
remote: Counting objects: 100% (400/400), done.
remote: Compressing objects: 100% (215/215), done.
remote: Total 400 (delta 209), reused 336 (delta 151), pack-reused 0 (from 0)
Receiving objects: 100% (400/400), 21.03 MiB | 30.20 MiB/s, done.
Resolving deltas: 100% (209/209), done.


In [2]:
from google.colab import drive
import sys
from pathlib import Path
drive.mount("/content/drive")

REPO_DIR = Path("/content/WasteScope")
sys.path.insert(0, str(REPO_DIR))

Mounted at /content/drive


In [3]:
import hashlib
import json
import shutil
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image

from ultralytics import YOLO
from src.data.annotations import read_yolo_labels
from src.data.boxes import xywhn_to_xyxy, xyxy_to_coco_xywh
from src.evaluation.metrics import evaluate_coco, coco_results, detection_errors

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
BASE = Path("/content/drive/MyDrive/WasteScope")
RUN_NAME = "D1_yolo26n_960"

VAL_REPORT = BASE / "reports/detection" / RUN_NAME / "val_960"
REPORT = BASE / "reports/detection" / RUN_NAME / "test_960"
METRICS = REPORT / "metrics"
FIGURES = REPORT / "figures"

WORK = Path("/content/wastescope_test") / RUN_NAME
DATA = WORK / "data"
BEST = WORK / "best.pt"

for folder in (WORK, DATA, REPORT, METRICS, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)

POLICY_PATH = VAL_REPORT / "confidence_tuning/selected_operating_point.json"

policy = json.loads(POLICY_PATH.read_text())
MANIFEST_PATH = VAL_REPORT / "video_split.json"
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8-sig"))

names = manifest["classes"]
IMGSZ = int(policy["imgsz"])

In [5]:
ready = DATA / ".test.ready"
if not ready.exists():
    local_zip = WORK / "test.zip"
    print("Copying test.zip to local storage...", flush=True)
    shutil.copy2(BASE / "prepared/detection/test.zip", local_zip)

    print("Extracting test.zip...", flush=True)
    with ZipFile(local_zip) as archive:
        archive.extractall(DATA)

    ready.write_text("Extraction complete", encoding="utf-8")
    local_zip.unlink()

image_paths = sorted((DATA / "test/images").glob("*.png"))

Copying test.zip to local storage...
Extracting test.zip...


In [6]:
(DATA / "data.yaml").write_text(
    yaml.safe_dump({
        "path": str(DATA),
        "train": None,
        "val": None,
        "test": "test/images",
        "names": dict(enumerate(names)),
    }, sort_keys=False), encoding="utf-8"
)

shutil.copy2(BASE / "runs/detect" / RUN_NAME / "weights/best.pt", BEST)
checkpoint_sha256 = hashlib.sha256(BEST.read_bytes()).hexdigest()

shutil.copy2(MANIFEST_PATH, REPORT / "video_split.json")
shutil.copy2(POLICY_PATH, REPORT / "operating_policy.json")

print("Test images:", len(image_paths))
print("Evaluation image size:", IMGSZ)

Test images: 5036
Evaluation image size: 960


In [7]:
model = YOLO(str(BEST))
captured = {}

def capture_test_predictions(validator):
    captured.update(
        predictions=list(validator.jdict),
        seen=int(validator.seen),
        half=bool(validator.args.half)
    )

model.add_callback("on_val_end", capture_test_predictions)

test_metrics = model.val(
    data=str(DATA / "data.yaml"),
    split="test",
    imgsz=IMGSZ,
    batch=16,
    device=0,
    workers=2,
    half=True,
    conf=0.001,
    iou=0.7,
    max_det=300,
    nms=False,
    agnostic_nms=False,
    augment=False,
    rect=True,
    cache=False,
    plots=True,
    save_json=True,
    project=str(REPORT.parent),
    name=REPORT.name,
    exist_ok=True
)

(REPORT / "native_predictions.json").write_text(
    json.dumps(captured["predictions"]), encoding="utf-8"
)

if captured["seen"] != len(image_paths):
    raise ValueError("Some test images were excluded. Inspect the evaluation warnings.")

overall = pd.DataFrame([{
    "Run": RUN_NAME,
    "Split": "test",
    "Precision": test_metrics.box.mp,
    "Recall": test_metrics.box.mr,
    "mAP50": test_metrics.box.map50,
    "mAP50-95": test_metrics.box.map
}])

overall.to_csv(METRICS / "overall_ultralytics.csv", index=False)
display(overall.round(4))

Ultralytics 8.4.1 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,376,201 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 185.0±51.7 MB/s, size: 2507.5 KB)
val: Scanning /content/wastescope_test/D1_yolo26n_960/data/test/labels... 5036 images, 2530 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5036/5036 88.6it/s 56.9s
val: New cache created: /content/wastescope_test/D1_yolo26n_960/data/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 315/315 1.1it/s 4:55
                   all       5036      10028      0.949      0.919      0.966      0.755
      container_yellow       1431       1475      0.947      0.868      0.944      0.733
       container_green       1380       1380      0.953      0.907      0.968      0.727
     container_default       2290       4171      0.964      0.933      0.977      0.791
        container_bl

,Run,Split,Precision,Recall,mAP50,mAP50-95
0,D1_yolo26n_960,test,0.9487,0.9193,0.9655,0.7554


In [8]:
images, annotations = [], []
filename_to_id = {}
prepared_class_map = {index: index for index in range(len(names))}

for image_id, path in enumerate(image_paths, start=1):
    with Image.open(path) as image:
        width, height = image.size

    images.append({
        "id": image_id,
        "file_name": path.name,
        "width": width,
        "height": height
    })
    filename_to_id[path.name] = image_id
    labels = read_yolo_labels(
        DATA / "test/labels" / f"{path.stem}.txt",
        class_map=prepared_class_map,
    )

    for label in labels:
        box = xyxy_to_coco_xywh(
            xywhn_to_xyxy(label.xywhn, width, height, clip=True)
        )
        annotations.append({
            "id": len(annotations) + 1,
            "image_id": image_id,
            "category_id": label.class_id + 1,
            "bbox": list(box),
            "area": box[2] * box[3],
            "iscrowd": 0
        })

ground_truth = {
    "images": images,
    "annotations": annotations,
    "categories": [
        {"id": index + 1, "name": name}
        for index, name in enumerate(names)
    ],
}

predictions = [{
    "image_id": filename_to_id[item["file_name"]],
    "category_id": item["category_id"],
    "bbox": item["bbox"],
    "score": item["score"],
} for item in captured["predictions"]]

for filename, content in {
    "coco_ground_truth.json": ground_truth,
    "coco_predictions.json": predictions,
}.items():
    (REPORT / filename).write_text(json.dumps(content), encoding="utf-8")

In [9]:
evaluation = evaluate_coco(ground_truth, predictions)
author_table = coco_results(evaluation)
author_table["Instances"] = author_table["Instances"].astype(int)
author_table.to_csv(METRICS / "per_class_coco.csv")


confidence_by_category = {
    category["id"]: policy["class_confidences"].get(
        category["name"], policy["default_confidence"]
    )
    for category in ground_truth["categories"]
}

operating_predictions = [
    item for item in predictions
    if item["score"] > confidence_by_category[item["category_id"]]
]

operating_evaluation = evaluate_coco(ground_truth, operating_predictions)
operating_errors = detection_errors(
    operating_evaluation, confidence=0.0, iou=policy["match_iou"]
)

per_class = operating_errors.per_class.copy()
per_class["F1"] = (
    2 * per_class["TP"]
    / (2 * per_class["TP"] + per_class["FP"] + per_class["FN"]).replace(0, np.nan)
)
per_class["Confidence"] = [
    policy["class_confidences"].get(name, policy["default_confidence"])
    for name in per_class.index
]

totals = per_class[["TP", "FP", "FN"]].sum()
operating_summary = pd.DataFrame([{
    "Run": RUN_NAME,
    "Split": "test",
    "Macro Precision": per_class["Precision"].fillna(0).mean(),
    "Macro Recall": per_class["Recall"].mean(),
    "Macro F1": per_class["F1"].mean(),
    "TP": int(totals["TP"]),
    "FP": int(totals["FP"]),
    "FN": int(totals["FN"]),
    "Oil Recall": per_class.loc["container_oil", "Recall"]
}])

per_class.to_csv(METRICS / "per_class_errors.csv")
operating_errors.per_image.to_csv(METRICS / "per_image_errors.csv")
operating_summary.to_csv(METRICS / "operating_metrics.csv", index=False)

for path in REPORT.iterdir():
    if path.is_file() and path.suffix.lower() in {".png", ".jpg"}:
        shutil.move(str(path), str(FIGURES / path.name))

creating index...
index created!
Loading and preparing results...
DONE (t=0.03s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=4.76s).
Accumulating evaluation results...
DONE (t=0.71s).
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=2.77s).
Accumulating evaluation results...
DONE (t=0.56s).


In [10]:
configuration = {
    "run": RUN_NAME,
    "model": "YOLO26n",
    "split": "test",
    "status": "complete",
    "imgsz": IMGSZ,
    "images": len(images),
    "instances": len(annotations),
    "classes": names,
    "checkpoint_sha256": checkpoint_sha256,
    "manifest_sha256": hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest(),
    "confidence_for_ap": 0.001,
    "nms": False,
    "prediction_max_det": 300,
    "coco_max_dets": [1, 10, 100],
    "half": captured["half"],
    "gpu": torch.cuda.get_device_name(0),
    "operating_policy": policy,
    "versions": {
        package: version(package)
        for package in ("ultralytics", "torch", "pycocotools", "numpy", "pandas")
    },
}

figure_links = "\n".join(
    f"- [{path.name}](figures/{path.name})"
    for path in sorted(FIGURES.iterdir()) if path.is_file()
)

readme = f"""# {RUN_NAME} — final test
Model: YOLO26n | Image size: {IMGSZ}
Images: {len(images)} | Instances: {len(annotations)}

## Native Ultralytics metrics

{overall.round(4).to_markdown(index=False)}

Native precision/recall use Ultralytics' operating-point calculation.
The fixed-policy metrics below are reported separately.

## Per-class COCO AP/AR

{author_table.round(4).to_markdown()}

## Fixed operating policy

Default confidence > {policy['default_confidence']};
Oil confidence > {policy['class_confidences']['container_oil']}.

{operating_summary.round(4).to_markdown(index=False)}

{per_class.round(4).to_markdown()}

## Protocol

- Checkpoint: best.pt; policy selected on validation and unchanged on test.
- AP predictions: confidence 0.001.
- COCO AP/AR: IoU 0.50:0.05:0.95, maxDets=[1,10,100].
- Operating errors: COCO class-aware, score-ordered matching, IoU 0.50.
- Native confusion matrix uses confidence > 0.25 and matching IoU > 0.45;
  its matching protocol differs from the operating-error tables.
- Single-run results; no mean ± std.

## Figures

{figure_links}

## Saved outputs

- [Metrics CSV files](metrics/)
- [Frozen operating policy](operating_policy.json)
- [Evaluation configuration](evaluation_config.json)
- [Frozen video split](video_split.json)
"""

(REPORT / "README.md").write_text(readme, encoding="utf-8")
(REPORT / "evaluation_config.json").write_text(
    json.dumps(configuration, indent=2), encoding="utf-8"
)

display(author_table.round(4))
display(operating_summary.round(4))
display(per_class.round(4))
print("Final test report saved:", REPORT)

,Instances,AP@0.50,AP@[.50:.95],AR@[.50:.95]
Class,,,,
container_yellow,1475,0.9305,0.6966,0.7557
container_green,1380,0.9592,0.6908,0.7707
container_default,4171,0.9632,0.7511,0.7887
container_blue,1635,0.9657,0.7549,0.8125
container_oil,182,0.9532,0.7044,0.7918
container_battery,600,0.9669,0.7511,0.8433
container_biodegradable,585,0.9651,0.7370,0.7891
All Classes,10028,0.9577,0.7266,0.7931


,Run,Split,Macro Precision,Macro Recall,Macro F1,TP,FP,FN,Oil Recall
0,D1_yolo26n_960,test,0.9004,0.9498,0.9221,9474,808,554,1.0


,TP,FP,FN,Precision,Recall,F1,Confidence
Class,,,,,,,
container_yellow,1352,126,123,0.9147,0.9166,0.9157,0.35
container_green,1298,95,82,0.9318,0.9406,0.9362,0.35
container_default,3988,240,183,0.9432,0.9561,0.9496,0.35
container_blue,1527,116,108,0.9294,0.9339,0.9317,0.35
container_oil,182,33,0,0.8465,1.0000,0.9169,0.20
container_battery,589,185,11,0.7610,0.9817,0.8574,0.35
container_biodegradable,538,13,47,0.9764,0.9197,0.9472,0.35


Final test report saved: /content/drive/MyDrive/WasteScope/reports/detection/D1_yolo26n_960/test_960
